In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import exp
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import scale
from sklearn import preprocessing
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import csv

## Setup
Getting info and exploring the data

In [ ]:
filename = 'mini-itc-data.csv'

df = pd.read_csv(filename)
print('done reading')

df.info()
df.head()

done reading
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   system_name                    100 non-null    object 
 1   region                         100 non-null    object 
 2   attack_type                    100 non-null    object 
 3   data_sensitivity_level         100 non-null    int64  
 4   records_exposed                100 non-null    int64  
 5   estimated_cost_per_record_usd  100 non-null    float64
 6   estimated_total_cost_usd       100 non-null    float64
 7   detection_delay_days           100 non-null    int64  
 8   response_time_days             100 non-null    int64  
 9   notification_required          100 non-null    object 
dtypes: float64(2), int64(4), object(4)
memory usage: 7.9+ KB


,system_name,region,attack_type,data_sensitivity_level,records_exposed,estimated_cost_per_record_usd,estimated_total_cost_usd,detection_delay_days,response_time_days,notification_required
0,CRM,latam-south2,External Hacker,4,56681,17.91,1014973.29,18,7,Yes
1,CRM,ca-central1,Misconfiguration,4,97901,18.07,1769502.68,10,9,Yes
2,Support,africa-south1,Misconfiguration,3,16274,13.34,217074.77,3,3,Yes
3,Billing,asia-south1,Insider,5,41640,21.44,892759.36,9,4,Yes
4,HR,eu-north1,Misconfiguration,5,45484,21.09,959464.03,7,7,Yes


Example Code

In [ ]:
dsl_avg = df["data_sensitivity_level"].sum()/len(df)
print("The average data sensitivity level is:",dsl_avg)

The average data sensitivity level is: 3.62


Analysis 1: Track Damage

In [ ]:
# Damage tracking metrics
total_records_exposed = df['records_exposed'].sum()
total_estimated_cost = df['estimated_total_cost_usd'].sum()
avg_cost_per_record = df['estimated_cost_per_record_usd'].mean()
avg_sensitivity = df['data_sensitivity_level'].mean()
notifications_required = df['notification_required'].value_counts()

# Print statements
print(f"Total Records Exposed: {total_records_exposed:,}")
print(f"Total Estimated Cost (USD): ${total_estimated_cost:,.2f}")
print(f"Average Cost per Record (USD): ${avg_cost_per_record:.2f}")
print(f"Average Data Sensitivity Level: {avg_sensitivity:.2f}")
print("Notification Required Counts:", notifications_required.to_dict())

Total Records Exposed: 4,349,853
Total Estimated Cost (USD): $73,615,882.35
Average Cost per Record (USD): $15.35
Average Data Sensitivity Level: 3.62
Notification Required Counts: {'Yes': 86, 'No': 14}


In [ ]:
# Group by Attack Type
cost_by_attack = df.groupby('attack_type')['estimated_total_cost_usd'].sum()

print("💥 Damage by Attack Type:")
for attack, cost in cost_by_attack.items():
    print(f"{attack:15} ${cost:,.2f}")
print("\n")  # space between sections

# Group by System
cost_by_system = df.groupby('system_name')['estimated_total_cost_usd'].sum()

print("🖥️ Damage by System:")
for system, cost in cost_by_system.items():
    print(f"{system:15} ${cost:,.2f}")
print("\n")  # space between sections

# Group by Region
cost_by_region = df.groupby('region')['estimated_total_cost_usd'].sum()

print("🌍 Damage by Region:")
for region, cost in cost_by_region.items():
    print(f"{region:20} ${cost:,.2f}")
print("\n")

💥 Damage by Attack Type:
External Hacker $11,582,888.61
Insider         $9,175,242.24
Misconfiguration $52,857,751.50


🖥️ Damage by System:
Analytics       $2,716,835.05
Billing         $28,724,234.86
CRM             $16,456,266.25
HR              $19,668,537.43
Support         $6,050,008.76


🌍 Damage by Region:
africa-south1        $1,440,033.04
ap-northeast1        $4,425,203.97
ap-northeast2        $3,675,220.11
ap-south1            $4,115,902.05
ap-south2            $2,313,768.59
ap-southeast1        $1,707,469.70
ap-southeast2        $2,435,676.61
asia-east1           $2,484,968.72
asia-south1          $1,047,131.86
ca-central1          $3,389,798.08
eu-central1          $2,206,584.14
eu-central2          $2,090,194.33
eu-north1            $3,727,167.99
eu-west1             $2,679,664.17
eu-west2             $3,155,714.14
latam-north1         $5,373,670.25
latam-south1         $3,745,727.08
latam-south2         $2,846,132.53
me-central1          $3,385,428.90
us-central1        